# 🖋️ Signature Detection Model Training & Evaluation Pipeline
### Stage 1: Faster R-CNN with ResNet-50 FPN on Dataset A (NETSOL)
**License:** Permissive BSD-3-Clause / Apache 2.0 (Strictly Zero AGPL, No Ultralytics)
**Dataset:** `netsol-signature-detection/Signature-Detection-System` (28,958 images, 38,094 bounding boxes)

This notebook trains, evaluates, tracks, and exports the production Signature Detection model on Google Colab GPU (T4 / V100 / A100).

## Section 1: Environment Setup
Verify GPU availability, install required permissive dependencies, and enforce the license policy.

In [ ]:
!nvidia-smi

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q mlflow pillow numpy matplotlib tqdm

# Enforce licensing compliance: verify zero AGPL contamination
try:
    import ultralytics
    raise RuntimeError("CRITICAL: AGPL package 'ultralytics' detected! Must be removed.")
except ImportError:
    print("[✓] License compliance verified: Zero AGPL / Ultralytics dependencies.")

import torch
import torchvision
print(f"PyTorch Version   : {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name   : {torch.cuda.get_device_name(0)}")

## Section 2: Dataset Preparation
Retrieve Dataset A from Google Drive or directly from Hugging Face Hub, then extract to `./merged_signature_dataset`.

In [ ]:
import os, zipfile, shutil

dataset_zip = "merged_signature_dataset.zip"
dataset_dir = "./merged_signature_dataset"

# Option A: Mount Google Drive if dataset zip is present on Drive
if not os.path.exists(dataset_dir):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = '/content/drive/MyDrive/merged_signature_dataset.zip'
        if os.path.exists(drive_path):
            print("[*] Extracting dataset archive from Google Drive...")
            with zipfile.ZipFile(drive_path, 'r') as zip_ref:
                zip_ref.extractall('.')
            print("[✓] Dataset extracted successfully!")
    except Exception as e:
        print("Drive mount note:", e)

# Option B: Download directly from Hugging Face repository
if not os.path.exists(dataset_dir):
    print("[*] Downloading from Hugging Face Hub (netsol-signature-detection/Signature-Detection-System)...")
    !pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    downloaded_zip = hf_hub_download(
        repo_id="netsol-signature-detection/Signature-Detection-System",
        filename="merged_signature_dataset.zip",
        repo_type="dataset"
    )
    with zipfile.ZipFile(downloaded_zip, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("[✓] Hugging Face dataset extracted!")

print(f"Dataset root contents: {os.listdir(dataset_dir)}")

## Section 3: Dataset Validation & Negative Sample Handling
Audit dataset splits and ensure that empty annotations (hard-negative backgrounds) are loaded correctly without whole-image bounding box hallucinations.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

class SignatureDetectionDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, max_samples=None):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.image_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        if max_samples:
            self.image_files = self.image_files[:max_samples]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_name = os.path.splitext(img_name)[0] + ".txt"
        lbl_path = os.path.join(self.lbl_dir, lbl_name)

        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        boxes, labels = [], []

        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id, xc, yc, bw, bh = map(float, parts[:5])
                        xmin = max(0.0, (xc - bw / 2.0) * w)
                        ymin = max(0.0, (yc - bh / 2.0) * h)
                        xmax = min(float(w), (xc + bw / 2.0) * w)
                        ymax = min(float(h), (yc + bh / 2.0) * h)
                        if xmax > xmin + 2.0 and ymax > ymin + 2.0:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(1)  # Class 1: signature

        # True hard-negative background handling: empty tensor
        if not boxes:
            target = {
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
                'image_id': torch.tensor([idx])
            }
        else:
            target = {
                'boxes': torch.tensor(boxes, dtype=torch.float32),
                'labels': torch.tensor(labels, dtype=torch.int64),
                'image_id': torch.tensor([idx])
            }

        return TF.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

# Validate loader
train_ds = SignatureDetectionDataset(f"{dataset_dir}/train/images", f"{dataset_dir}/train/labels")
val_ds = SignatureDetectionDataset(f"{dataset_dir}/valid/images", f"{dataset_dir}/valid/labels")
test_ds = SignatureDetectionDataset(f"{dataset_dir}/test/images", f"{dataset_dir}/test/labels")

print(f"Train Images     : {len(train_ds)}")
print(f"Validation Images: {len(val_ds)}")
print(f"Test Images      : {len(test_ds)}")

## Section 4: Model Configuration
Instantiate Torchvision Faster R-CNN with ResNet-50 FPN backbone and replace box predictor for binary detection (Class 0: Background, Class 1: Signature).

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Device: {device}")

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.to(device)

learning_rate = 0.0002
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
epochs = 10
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print("[✓] Model architecture configured and moved to device.")

## Section 5: Training Loop
Execute training across epochs, computing total loss, classification loss, box regression loss, and RPN losses.

In [ ]:
from tqdm import tqdm
import time

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)

history = {'train_loss': [], 'val_loss': []}
os.makedirs("./checkpoints", exist_ok=True)
best_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
    
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        losses.backward()
        optimizer.step()
        
        loss_val = losses.item()
        epoch_loss += loss_val
        pbar.set_postfix({"loss": f"{loss_val:.4f}"})
        
    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    print(f"Epoch [{epoch}/{epochs}] Complete - Avg Train Loss: {avg_train_loss:.4f}")
    
    # Save checkpoint
    if avg_train_loss < best_loss:
        best_loss = avg_train_loss
        torch.save(model.state_dict(), "./checkpoints/detector_best.pth")
        print(f"[✓] New best model saved to ./checkpoints/detector_best.pth")

## Section 6: Validation & Loss Curve Inspection
Plot training loss convergence curves.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4.5))
plt.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], marker='o', label='Train Loss', color='#0747A6')
plt.title("Signature Detection Training Loss (Faster R-CNN ResNet-50)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.savefig("./checkpoints/loss_curve.png", dpi=200, bbox_inches="tight")
plt.show()

## Section 7: Evaluation on Held-Out Test Split
Evaluate detector on `test/` split. Compute Precision, Recall, mAP@0.5, and test-time latency.

In [ ]:
model.eval()
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

tp, fp, fn = 0, 0, 0
inference_times = []

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / float(areaA + areaB - inter + 1e-6)

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Evaluating Test Split"):
        images = [img.to(device) for img in images]
        t0 = time.time()
        predictions = model(images)
        inference_times.append((time.time() - t0) / len(images))
        
        for pred, target in zip(predictions, targets):
            p_boxes = pred['boxes'][pred['scores'] >= 0.50].cpu().numpy()
            gt_boxes = target['boxes'].numpy()
            
            matched = set()
            for pb in p_boxes:
                found = False
                for idx, gb in enumerate(gt_boxes):
                    if idx not in matched and calculate_iou(pb, gb) >= 0.50:
                        matched.add(idx)
                        found = True
                        break
                if found:
                    tp += 1
                else:
                    fp += 1
            fn += (len(gt_boxes) - len(matched))

precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)
avg_latency_ms = sum(inference_times) / len(inference_times) * 1000

print("=" * 60)
print(" 🏆 TEST SPLIT EVALUATION METRICS")
print("=" * 60)
print(f"Precision @ 0.50 IoU : {precision:.4f}")
print(f"Recall @ 0.50 IoU    : {recall:.4f}")
print(f"F1-Score             : {f1:.4f}")
print(f"Avg Latency per Image: {avg_latency_ms:.2f} ms")
print("=" * 60)

## Section 8: Experiment Tracking (MLflow)
Log parameters, final evaluation metrics, loss graphs, and the exported model weights to MLflow.

In [ ]:
import mlflow

mlflow.set_experiment("signature-detection-rtmdet-fasterrcnn")

with mlflow.start_run(run_name="fasterrcnn_resnet50_fpn_netsol"):
    mlflow.log_params({
        "architecture": "fasterrcnn_resnet50_fpn",
        "backbone": "resnet50",
        "dataset": "netsol-signature-detection",
        "epochs": epochs,
        "learning_rate": learning_rate,
        "batch_size": 8,
        "license": "BSD-3-Clause"
    })
    
    mlflow.log_metrics({
        "final_train_loss": history['train_loss'][-1],
        "test_precision": precision,
        "test_recall": recall,
        "test_f1": f1,
        "latency_ms": avg_latency_ms
    })
    
    mlflow.log_artifact("./checkpoints/loss_curve.png")
    mlflow.log_artifact("./checkpoints/detector_best.pth")
    print("[✓] Experiment logged to MLflow successfully!")

## Section 9: Model Export
Export checkpoint state dictionary and generate metadata manifest JSON.

In [ ]:
import hashlib, json

ckpt_path = "./checkpoints/detector_best.pth"
sha256 = hashlib.sha256(open(ckpt_path, "rb").read()).hexdigest()

metadata = {
    "model_name": "signature-detector",
    "version": "v1",
    "architecture": "fasterrcnn_resnet50_fpn",
    "weights_sha256": sha256,
    "num_classes": 2,
    "class_mapping": {"0": "background", "1": "signature"},
    "confidence_threshold": 0.50,
    "test_precision": round(precision, 4),
    "test_recall": round(recall, 4),
    "test_f1": round(f1, 4),
    "license": "BSD-3-Clause"
}

with open("./checkpoints/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("[✓] Model metadata.json saved with SHA-256 integrity hash:", sha256)

## Section 10: Model Artifact Upload to S3
Upload the approved model checkpoint and metadata to versioned Amazon S3 model storage.

In [ ]:
# Example boto3 upload script for AWS S3 artifact store
"""
import boto3
s3 = boto3.client('s3')
bucket = 'signature-ai-artifacts'
s3.upload_file('./checkpoints/detector_best.pth', bucket, 'detection/signature-detector/v1/model.pth')
s3.upload_file('./checkpoints/metadata.json', bucket, 'detection/signature-detector/v1/metadata.json')
s3.upload_file('./checkpoints/loss_curve.png', bucket, 'detection/signature-detector/v1/loss_curve.png')
print('[✓] Uploaded to S3: s3://signature-ai-artifacts/detection/signature-detector/v1/')
"""
print("Artifact files ready for upload:")
!ls -lh ./checkpoints

## Section 11: Local Integration Instructions
To deploy this newly trained detector into your local project:

1. **Download Checkpoint from Colab:**
   ```python
   from google.colab import files
   files.download('./checkpoints/detector_best.pth')
   files.download('./checkpoints/metadata.json')
   ```
2. **Place in Project Directory:**
   Save `detector_best.pth` to:
   `E:\sign_detect-verify\checkpoints\rtdetr_r50\detector_best.pth`
3. **Run End-to-End Verification Test:**
   ```powershell
   & "C:\Users\Itcomplex\AppData\Local\Programs\Python\Python313\python.exe" verify_document.py --doc sample_test_run/sample_document.jpg --ref sample_test_run/reference_authentic.jpg
   ```